# Resource Recommender — Model Exploration

This notebook explores the AI/ML models used in the framework:
1. Isolation Forest for anomaly detection
2. Linear regression for predictive scaling
3. Rule engine evaluation

**Dissertation reference**: Section 5 — Learning and Decision Mechanism

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.ensemble import IsolationForest
from sklearn.metrics import precision_score, recall_score
import warnings
warnings.filterwarnings('ignore')

# Load sample deployment data
df = pd.read_csv('training_data/deploy_logs_sample.csv', parse_dates=['timestamp'])
print(f'Loaded {len(df)} deployment records')
df.head()

## 1. Anomaly Detection — Isolation Forest

Training on 5 features: duration, resources added/changed/destroyed, plan size.

In [ ]:
features = ['duration_seconds', 'resources_added', 'resources_changed',
            'resources_destroyed', 'plan_size_bytes']

X = df[features].fillna(0)

# Ground truth: failures are 'true' anomalies
y_true = (df['status'] == 'failure').astype(int)

model = IsolationForest(contamination=0.05, random_state=42, n_estimators=100)
preds = model.fit_predict(X)
scores = model.score_samples(X)

y_pred = (preds == -1).astype(int)

precision = precision_score(y_true, y_pred, zero_division=0)
recall    = recall_score(y_true, y_pred, zero_division=0)

print(f'Precision: {precision:.2f} (target >= 0.80)')
print(f'Recall:    {recall:.2f} (target >= 0.75)')
print(f'Anomalies flagged: {y_pred.sum()} / {len(y_pred)}')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Plot 1: Duration vs anomaly score
scatter = axes[0].scatter(df['duration_seconds'], scores,
                          c=y_pred, cmap='RdYlGn_r', alpha=0.7, s=80)
axes[0].set_xlabel('Duration (seconds)')
axes[0].set_ylabel('Anomaly score (lower = more anomalous)')
axes[0].set_title('Isolation Forest: Duration vs Anomaly Score')
plt.colorbar(scatter, ax=axes[0], label='0=normal, 1=anomaly')

# Plot 2: Resources added vs destroyed
scatter2 = axes[1].scatter(df['resources_added'], df['resources_destroyed'],
                           c=y_pred, cmap='RdYlGn_r', alpha=0.7, s=80)
axes[1].set_xlabel('Resources added')
axes[1].set_ylabel('Resources destroyed')
axes[1].set_title('Resource Change Profile')
plt.colorbar(scatter2, ax=axes[1], label='0=normal, 1=anomaly')

plt.tight_layout()
plt.savefig('training_data/anomaly_visualisation.png', dpi=150, bbox_inches='tight')
plt.show()
print('Plot saved.')

## 2. Predictive Scaling — Linear Regression

In [ ]:
import math

# Synthetic 7-day hourly CPU data
hours = 168
cpu_data = [
    30 + 15 * math.sin(2 * math.pi * i / 24) + 0.05 * i + (hash(str(i)) % 7 - 3)
    for i in range(hours)
]

# Linear regression (no numpy required — pure Python)
n = len(cpu_data)
x_mean = (n - 1) / 2
y_mean = sum(cpu_data) / n
ss_xx  = sum((i - x_mean) ** 2 for i in range(n))
ss_xy  = sum((i - x_mean) * (v - y_mean) for i, v in enumerate(cpu_data))
slope  = ss_xy / ss_xx
intercept = y_mean - slope * x_mean

# Forecast 24h ahead
forecast = intercept + slope * (n + 24)
forecast = max(0, min(100, forecast))

print(f'7-day CPU mean: {y_mean:.1f}%')
print(f'Trend (slope):  {slope:.4f}% per hour')
print(f'24h forecast:   {forecast:.1f}%')

# Plot
fig, ax = plt.subplots(figsize=(12, 4))
ax.plot(range(hours), cpu_data, alpha=0.7, label='Historical CPU %')
future_x = range(hours, hours + 25)
future_y = [intercept + slope * x for x in future_x]
ax.plot(future_x, future_y, 'r--', label=f'Forecast (24h = {forecast:.1f}%)')
ax.axhline(65, color='orange', linestyle=':', label='Scale-up threshold (65%)')
ax.axhline(25, color='green',  linestyle=':', label='Scale-down threshold (25%)')
ax.set_xlabel('Hour')
ax.set_ylabel('CPU utilisation (%)')
ax.set_title('Predictive Scaling: Linear Regression Forecast')
ax.legend()
plt.tight_layout()
plt.savefig('training_data/scaling_forecast.png', dpi=150, bbox_inches='tight')
plt.show()

## 3. Evaluation summary

In [ ]:
# Compute framework KPIs from sample data
total       = len(df)
failures    = (df['status'] == 'failure').sum()
error_rate  = 100 * failures / total
avg_dur_min = df['duration_seconds'].mean() / 60
baseline_min = 120
time_reduction = 100 * (baseline_min - avg_dur_min) / baseline_min

print('=== Framework KPIs ===')
print(f'Total deployments:        {total}')
print(f'Avg duration:             {avg_dur_min:.1f} min (baseline: {baseline_min} min)')
print(f'Deployment time reduction: {time_reduction:.1f}%')
print(f'Error rate:               {error_rate:.1f}% (baseline: 18%)')
print(f'Error rate improvement:   {18 - error_rate:.1f} pp')
print(f'Anomaly precision:        {precision:.2f}')
print(f'Anomaly recall:           {recall:.2f}')